# 04 — Release identifiability and flow-map v1.1

Questo notebook misura l'ambiguità introdotta dal rilascio sinaptico probabilistico e confronta **B0 persistence**, **B1 ridge per blocchi** e il baseline **B3 structured shared residual** congelato da 02b. Non implementa HayFlow-Hines.

Il dataset è sempre un composito logico: base e top-up restano due HDF5 separati. L'unica autorità è `composite_dataset_manifest.json`; lo shard supplementare rimane esclusivamente in validation. `U_realized` viene ricostruito causalmente unendo schedule e record di rilascio, inclusi i fallimenti, senza leggere `S_(t+1)`.

## 1. Checkout coerente e dipendenze

Usare notebook, librerie e configurazione dello stesso checkout. `diagnostic_full` è l'unico profilo decisionale; `smoke` serve soltanto a verificare l'esecuzione.

In [ ]:
import os, subprocess, sys
from pathlib import Path

WORKSPACE = Path('/kaggle/working/hayflow_workspace')
ELM_REPO = WORKSPACE / 'elmneuron'
ELM_REF = os.environ.get('HAYFLOW_ELM_REF', 'main')
WORKSPACE.mkdir(parents=True, exist_ok=True)
if not ELM_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Zagred47/giada.git', str(ELM_REPO)], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'fetch', 'origin', ELM_REF], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
REVISION = subprocess.check_output(['git', '-C', str(ELM_REPO), 'rev-parse', 'HEAD'], text=True).strip()
sys.path.insert(0, str(ELM_REPO))
print('Revisione coerente caricata:', REVISION)

In [ ]:
import h5py, numpy, pandas, pyarrow, torch, yaml
print({'h5py': h5py.__version__, 'numpy': numpy.__version__, 'pandas': pandas.__version__, 'pyarrow': pyarrow.__version__, 'torch': torch.__version__, 'cuda': torch.cuda.is_available()})

## 2. Risoluzione del manifest composito

Aggiungere come input Kaggle il dataset base e lo ZIP `hayflow_bap_validation_support_topup_v3.zip`. Se Kaggle monta il base come un singolo `archive.zip`, il loader estrae nella cache soltanto HDF5 e metadati necessari: non fonde gli shard.

In [ ]:
import zipfile

input_root = Path('/kaggle/input')
topup_override = os.environ.get('HAYFLOW_TOPUP_V3')
topup_candidates = [Path(topup_override).expanduser()] if topup_override else []
topup_candidates.extend(input_root.rglob('hayflow_bap_validation_support_topup_v3.zip'))
topup_candidates.extend(path.parent for path in input_root.rglob('composite_dataset_manifest.json'))
TOPUP_SOURCE = next((path.resolve() for path in topup_candidates if path.exists()), None)
assert TOPUP_SOURCE is not None, 'Top-up v3 non trovato negli input Kaggle.'

if TOPUP_SOURCE.is_file() and TOPUP_SOURCE.suffix.lower() == '.zip':
    TOPUP_ROOT = Path('/kaggle/working/hayflow04_topup')
    marker = TOPUP_ROOT / '.source_size'
    stamp = str(TOPUP_SOURCE.stat().st_size)
    if not marker.is_file() or marker.read_text().strip() != stamp:
        import shutil
        if TOPUP_ROOT.exists(): shutil.rmtree(TOPUP_ROOT)
        TOPUP_ROOT.mkdir(parents=True)
        root = TOPUP_ROOT.resolve()
        with zipfile.ZipFile(TOPUP_SOURCE) as archive:
            for member in archive.infolist():
                target = (TOPUP_ROOT / member.filename).resolve()
                assert root in target.parents or target == root, member.filename
            archive.extractall(TOPUP_ROOT)
        marker.write_text(stamp)
    COMPOSITE_MANIFEST = next(TOPUP_ROOT.rglob('composite_dataset_manifest.json'))
elif TOPUP_SOURCE.is_dir():
    COMPOSITE_MANIFEST = next(TOPUP_SOURCE.rglob('composite_dataset_manifest.json'))
else:
    COMPOSITE_MANIFEST = TOPUP_SOURCE

base_override = os.environ.get('HAYFLOW_TARGETED_BASE')
base_candidates = [Path(base_override).expanduser()] if base_override else []
base_candidates.append(Path('/kaggle/input/datasets/alessandrobelli/hayflow-targeted-transition-dataset-v1-1-base'))
base_candidates.extend(path.parent for path in input_root.rglob('transition_dataset.h5') if 'targeted-transition-dataset-v1-1-base' in str(path))
BASE_SOURCE = next((path.resolve() for path in base_candidates if path.exists()), None)
assert BASE_SOURCE is not None, 'Dataset base targeted v1.1 non trovato.'
print('Manifest composito:', COMPOSITE_MANIFEST)
print('Sorgente base:', BASE_SOURCE)

## 3. Verifica crittografica e loader shard-aware

L'hash del base da circa 6 GiB può richiedere qualche minuto. Il tracker mostra percentuale ed ETA; il controllo non viene saltato nel profilo decisionale.

In [ ]:
import time
from src.hayflow_data import prepare_composite_flowmap_bundle

hash_started = {}
hash_last = {}
def hash_progress(name, done, total):
    now = time.monotonic(); hash_started.setdefault(name, now)
    percent = int(100 * done / total)
    if percent >= hash_last.get(name, -5) + 5 or done == total:
        elapsed = now - hash_started[name]; rate = done / max(elapsed, 1e-9)
        eta = (total - done) / max(rate, 1e-9)
        print(f'[HayFlow 04][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min', flush=True)
        hash_last[name] = percent

bundle = prepare_composite_flowmap_bundle(
    COMPOSITE_MANIFEST, base_source=BASE_SOURCE,
    cache_dir=Path('/kaggle/working/hayflow04_cache'),
    verify_hashes=True, progress=hash_progress,
)
print({'fingerprint': bundle.fingerprint, 'transitions': bundle.transition_count, 'physical_merge': False})

## 4. Configurazione e preflight

La normalizzazione è stimata soltanto sul train con campionamento stratificato per episodio. Gli stati restano lazy negli HDF5. Il report deve confermare 369 episodi, 29.880 transizioni e 8 episodi top-up tutti in validation.

In [ ]:
from dataclasses import fields
from src.hayflow_model import ReleaseExperimentConfig, ReleaseIdentifiabilityExperiment

raw_config = yaml.safe_load((ELM_REPO / 'configs/hayflow/release_identifiability_flowmap_v1_1.yml').read_text())
raw_config.pop('schema_version', None)
for name in ('initialization_seeds', 'rollout_horizons_ms'):
    raw_config[name] = tuple(raw_config[name])
raw_config['profile'] = os.environ.get('HAYFLOW_04_PROFILE', raw_config['profile'])
config = ReleaseExperimentConfig(**raw_config)
OUTPUT_DIR = Path('/kaggle/working/artifacts/release_identifiability_flowmap_v1_1')
session = ReleaseIdentifiabilityExperiment(bundle, OUTPUT_DIR, config)
prepare_report = session.prepare()
display(prepare_report['loader'])
assert prepare_report['loader']['episode_count'] == 369
assert prepare_report['loader']['transition_count'] == 29880
assert prepare_report['loader']['topup_validation_only']
assert prepare_report['loader']['event_counts_coherent']

## 5. Esperimento completo

Questa è la cella lunga. Esegue identifiability, B0, tre B1, nove B3 primari (tre viste × tre seed), rollout 2/4/8/16/32 ms, branching, recovery e tre ablation aggiuntive dello stato. Ogni epoca e ogni run mostrano avanzamento ed ETA. I checkpoint sono ripresi soltanto se il fingerprint coincide esattamente.

In [ ]:
final_report = session.run()
display({
    'valid': final_report['valid'],
    'decision_grade': final_report['decision_grade'],
    'decision': final_report['decision'],
    'identifiability': final_report['synaptic_identifiability'],
    'events': final_report['event_fidelity'],
})

## 6. Controllo degli output

Il gate documentale richiede tutti i report e le tabelle preregistrate. Una classe positiva mai rilevata ha F1 pari a zero; gli shard non vengono mai riscritti.

In [ ]:
required = [
    'composite_loader_report.json', 'input_view_schema.json', 'identifiability_report.json',
    'normalization_schema.json', 'model_configs.json', 'seed_metrics.parquet',
    'one_step_metrics.parquet', 'rollout_metrics.parquet', 'event_metrics_pooled.parquet',
    'regional_drift.parquet', 'peak_attenuation.parquet', 'branching_metrics.parquet',
    'recovery_metrics.parquet', 'state_ablation_metrics.parquet', 'final_report.json',
]
missing = [name for name in required if not (OUTPUT_DIR / name).is_file()]
assert not missing, missing
print('Output completi:', len(required), '| figure:', len(list((OUTPUT_DIR / 'figures').glob('*.png'))))

## 7. Download nel browser

La cella usa il metodo compatibile con Kaggle già validato nel progetto: ZIP in `/kaggle/working`, Base64, JavaScript, `Blob` e click su un anchor temporaneo. Per impostazione predefinita include report, tabelle e figure ma non i checkpoint pesanti; impostare `HAYFLOW_DOWNLOAD_CHECKPOINTS=1` prima della cella per includerli.

In [ ]:
from pathlib import Path
from shutil import copytree, make_archive, rmtree
import base64, os
from IPython.display import Javascript, display

include_checkpoints = os.environ.get('HAYFLOW_DOWNLOAD_CHECKPOINTS', '0') == '1'
archive_source = OUTPUT_DIR
staging = Path('/kaggle/working/hayflow04_download')
if not include_checkpoints:
    if staging.exists(): rmtree(staging)
    copytree(OUTPUT_DIR, staging, ignore=lambda path, names: {'checkpoints'} if 'checkpoints' in names else set())
    archive_source = staging
zip_base = Path('/kaggle/working/hayflow_release_identifiability_flowmap_v1_1')
zip_path = Path(make_archive(str(zip_base), 'zip', root_dir=archive_source.parent, base_dir=archive_source.name))
encoded = base64.b64encode(zip_path.read_bytes()).decode('ascii')
filename = zip_path.name
display(Javascript(f'''
const binary = atob('{encoded}');
const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const anchor = document.createElement('a');
anchor.href = url; anchor.download = '{filename}';
document.body.appendChild(anchor); anchor.click(); anchor.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
'''))
print('Download avviato:', zip_path, f'({zip_path.stat().st_size / 2**20:.1f} MiB)', '| checkpoint inclusi:', include_checkpoints)